# PoC 2c: Observable RAG Pipeline

Every claim traceable to a source passage. Faithfulness checks. LLM-based surprise with explanation. Full audit trail.

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import AzureOpenAI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
EMBED_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-small")

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
RETRIEVAL_TOP_K = 8
EXTRACTION_BATCH_SIZE = 10

r = client.embeddings.create(input=["test"], model=EMBED_MODEL)
print(f"LLM: {MODEL}, Embeddings: {EMBED_MODEL} (dim={len(r.data[0].embedding)})")

## 1. Build Corpus via Citation Graph Traversal

No hardcoded queries. Start from seed papers, follow citations outward 2-3 hops. The research community defines relevance, not our priors.

In [ ]:
OA_BASE = "https://api.openalex.org"
OA_PARAMS = {"mailto": "paul.keck@campus.lmu.de"}  # polite pool = no rate limit

# STEEP-diverse seeds: not just technology, also policy, economics, environment, social
SEED_QUERIES = [
    # Technological
    "spectrum monitoring survey",
    "cognitive radio spectrum sensing",
    "6G terahertz communication",
    # Political / Regulatory
    "spectrum policy regulation 5G",
    "electronic warfare spectrum dominance",
    # Economic
    "telecom test measurement market disruption",
    "Open RAN economics vendor",
    # Social
    "digital divide wireless connectivity rural",
    "electromagnetic field exposure health 5G",
    # Environmental
    "energy efficiency green wireless network",
    "satellite constellation spectrum interference",
]

def oa_reconstruct_abstract(inv_index: dict | None) -> str | None:
    if not inv_index:
        return None
    words = {}
    for word, positions in inv_index.items():
        for pos in positions:
            words[pos] = word
    return " ".join(words[i] for i in sorted(words))

def oa_parse_work(work: dict) -> dict | None:
    abstract = oa_reconstruct_abstract(work.get("abstract_inverted_index"))
    if not abstract:
        return None
    return {
        "openalex_id": work["id"],
        "title": work.get("display_name", ""),
        "abstract": abstract,
        "year": work.get("publication_year"),
        "cited_by_count": work.get("cited_by_count", 0),
        "referenced_works": work.get("referenced_works", []),
        "pdf_url": next((loc.get("pdf_url") for loc in work.get("locations", []) if loc.get("pdf_url")), None),
    }

def oa_search(query: str, limit: int = 5) -> list[dict]:
    try:
        resp = requests.get(f"{OA_BASE}/works", params={**OA_PARAMS, "search": query, "per_page": limit}, timeout=10)
        resp.raise_for_status()
        return [p for p in (oa_parse_work(w) for w in resp.json().get("results", [])) if p]
    except Exception as e:
        print(f"  Search failed for '{query}': {e}")
        return []

def oa_get_works(openalex_ids: list[str], batch_size: int = 50) -> list[dict]:
    results = []
    for i in range(0, len(openalex_ids), batch_size):
        batch = openalex_ids[i:i+batch_size]
        id_filter = "|".join(batch)
        try:
            resp = requests.get(f"{OA_BASE}/works", params={**OA_PARAMS, "filter": f"openalex:{id_filter}", "per_page": batch_size}, timeout=15)
            resp.raise_for_status()
            for w in resp.json().get("results", []):
                parsed = oa_parse_work(w)
                if parsed:
                    results.append(parsed)
        except Exception:
            pass
        time.sleep(0.2)
    return results

def oa_get_citations(openalex_id: str, limit: int = 50) -> list[dict]:
    try:
        resp = requests.get(f"{OA_BASE}/works", params={**OA_PARAMS, "filter": f"cites:{openalex_id}", "per_page": limit, "sort": "cited_by_count:desc"}, timeout=10)
        resp.raise_for_status()
        return [p for p in (oa_parse_work(w) for w in resp.json().get("results", [])) if p]
    except Exception:
        return []

print("Finding seed papers...")
seed_ids = set()
all_oa_papers = {}

for query in SEED_QUERIES:
    results = oa_search(query, limit=3)
    for p in results:
        oid = p["openalex_id"]
        if oid not in all_oa_papers:
            all_oa_papers[oid] = {**p, "hop": 0, "discovered_via": f"seed query: {query}"}
            seed_ids.add(oid)
    print(f"  '{query}' → {len(results)} papers")
    time.sleep(0.2)

print(f"\nSeeds: {len(seed_ids)} papers from {len(SEED_QUERIES)} queries")
for oid in list(seed_ids)[:10]:
    print(f"  • {all_oa_papers[oid]['title'][:80]}")

In [ ]:
MAX_HOPS = 2
MAX_REFS_PER_PAPER = 20
MAX_CITES_PER_PAPER = 20
MAX_CORPUS = 500

for hop in range(1, MAX_HOPS + 1):
    frontier = [oid for oid, p in all_oa_papers.items() if p["hop"] == hop - 1]
    print(f"\nHop {hop}: expanding from {len(frontier)} papers...")
    new_count = 0

    for i, oid in enumerate(frontier):
        if len(all_oa_papers) >= MAX_CORPUS * 2:
            print(f"  Early stop at {len(all_oa_papers)} papers (enough candidates)")
            break

        parent_title = all_oa_papers[oid]["title"][:60]

        ref_ids = [r for r in all_oa_papers[oid].get("referenced_works", []) if r not in all_oa_papers][:MAX_REFS_PER_PAPER]
        if ref_ids:
            refs = oa_get_works(ref_ids)
            for p in refs:
                rpid = p["openalex_id"]
                if rpid not in all_oa_papers:
                    all_oa_papers[rpid] = {**p, "hop": hop, "discovered_via": f"cited by '{parent_title}...'"}
                    new_count += 1

        cites = oa_get_citations(oid, limit=MAX_CITES_PER_PAPER)
        for p in cites:
            cpid = p["openalex_id"]
            if cpid not in all_oa_papers:
                all_oa_papers[cpid] = {**p, "hop": hop, "discovered_via": f"cites '{parent_title}...'"}
                new_count += 1
        time.sleep(0.3)

        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(frontier)}] {new_count} new papers so far...")

    print(f"  Hop {hop} done: +{new_count} papers (total: {len(all_oa_papers)})")

# Smart pruning: keep seeds + hop 1, then from hop 2 prefer RECENT + LOW-CITED papers (weak signals)
if len(all_oa_papers) > MAX_CORPUS:
    core = {oid: p for oid, p in all_oa_papers.items() if p["hop"] <= 1}
    outer = [(oid, p) for oid, p in all_oa_papers.items() if p["hop"] >= 2]

    # Score: recent papers with moderate citations = weak signals
    # Heavily-cited = mainstream (boring). Zero-cited = possibly junk.
    # Sweet spot: recent + some citations but not thousands
    import math
    current_year = 2026
    for oid, p in outer:
        year = p.get("year") or 2020
        recency = max(0, 1 - (current_year - year) / 15)  # 0-1, newer = higher
        cites = p.get("cited_by_count", 0)
        # Log-scaled citation penalty: diminishing returns, penalize >500 citations
        cite_score = math.log1p(min(cites, 100)) / math.log1p(100)  # caps at 100
        mainstream_penalty = max(0, (cites - 200) / 1000)  # penalize heavily-cited
        p["_signal_score"] = recency * 0.5 + cite_score * 0.3 - mainstream_penalty * 0.2

    outer.sort(key=lambda x: x[1].get("_signal_score", 0), reverse=True)
    slots = MAX_CORPUS - len(core)
    for oid, p in outer[:max(slots, 0)]:
        core[oid] = p

    # Stats
    kept_outer = [p for oid, p in outer[:max(slots, 0)]]
    avg_year = np.mean([p.get("year", 2020) for p in kept_outer]) if kept_outer else 0
    avg_cites = np.mean([p.get("cited_by_count", 0) for p in kept_outer]) if kept_outer else 0
    print(f"\nPruned: {len(all_oa_papers)} → {len(core)} papers")
    print(f"  Hop 2+ selection: avg year={avg_year:.0f}, avg citations={avg_cites:.0f} (weak signal bias)")
    all_oa_papers = core

papers = []
for oid, p in all_oa_papers.items():
    papers.append({
        "id": oid,
        "title": p["title"],
        "abstract": p["abstract"],
        "published": str(p.get("year", "unknown")),
        "pdf_url": p.get("pdf_url"),
        "source": "openalex",
        "hop": p["hop"],
        "discovered_via": p["discovered_via"],
    })

print(f"\nTotal corpus: {len(papers)} papers")
hop_counts = pd.Series([p["hop"] for p in papers]).value_counts().sort_index()
for hop, count in hop_counts.items():
    print(f"  Hop {hop}: {count} papers {'(seeds)' if hop == 0 else ''}")

### Corpus Bias Disclosure

The corpus boundary is defined by seed queries + citation graph expansion. Seed queries encode our priors. Citation hops reduce but don't eliminate this bias — papers disconnected from the seed graph are invisible.

In [ ]:
hop_df = pd.DataFrame(papers)
print(f"Corpus: {len(papers)} papers total\n")

print("Hop distribution:")
hop_counts = hop_df["hop"].value_counts().sort_index()
for hop, count in hop_counts.items():
    pct = count / len(papers) * 100
    bar = "█" * int(pct / 2)
    label = "(seeds)" if hop == 0 else ""
    print(f"  Hop {hop} {label:>8}: {count:4d} papers ({pct:4.1f}%) {bar}")

print(f"\nSeed queries ({len(SEED_QUERIES)}):")
for q in SEED_QUERIES:
    seed_count = sum(1 for p in papers if p["discovered_via"] == f"seed query: {q}")
    print(f"  • \"{q}\" → {seed_count} seed papers")

pdf_available = sum(1 for p in papers if p.get("pdf_url"))
print(f"\nPDF links available: {pdf_available}/{len(papers)}")
print(f"\n⚠ Bias: {len(SEED_QUERIES)} seed queries define the starting boundary.")
print(f"  Citation graph expanded the corpus, but papers outside this graph are invisible.")

## 2. Download PDFs + Extract Full Text

In [ ]:
import pymupdf

PDF_DIR = Path("../data/pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

def download_pdf(url: str, path: Path, timeout: int = 30) -> bool:
    try:
        resp = requests.get(url, timeout=timeout, stream=True)
        resp.raise_for_status()
        with open(path, "wb") as f:
            for chunk in resp.iter_content(8192):
                f.write(chunk)
        return True
    except Exception:
        return False

def extract_text_from_pdf(path: Path, max_pages: int = 15) -> str | None:
    try:
        doc = pymupdf.open(str(path))
        text = "".join(page.get_text() for page in doc[:max_pages])
        doc.close()
        return text.strip() if len(text.strip()) > 200 else None
    except Exception:
        return None

pdf_ok, pdf_fail, abstract_only = 0, 0, 0
for i, paper in enumerate(papers):
    if i % 50 == 0:
        print(f"Processing {i}/{len(papers)}...")
    if paper.get("pdf_url"):
        safe_name = paper["id"].replace("/", "_").replace(":", "_")[-40:] + ".pdf"
        pdf_path = PDF_DIR / safe_name
        if not pdf_path.exists():
            download_pdf(paper["pdf_url"], pdf_path)
        text = extract_text_from_pdf(pdf_path) if pdf_path.exists() else None
        if text:
            paper["full_text"] = text
            pdf_ok += 1
            continue
        pdf_fail += 1
    paper["full_text"] = f"Title: {paper['title']}\n\nAbstract: {paper['abstract']}"
    abstract_only += 1

print(f"\nPDF: {pdf_ok} ok, {pdf_fail} failed → abstract fallback. Abstract only: {abstract_only}")

## 3. Chunk + Index (with provenance metadata)

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

Settings.embed_model = AzureOpenAIEmbedding(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_deployment=EMBED_MODEL,
)
Settings.llm = None

documents = [
    Document(
        text=p["full_text"],
        metadata={"title": p["title"], "published": p["published"], "source": p["source"], "paper_id": p["id"], "paper_index": i},
    )
    for i, p in enumerate(papers)
]

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection("poc2c")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_documents(documents, vector_store=vector_store, transformations=[splitter], show_progress=True)
print(f"\nIndexed {len(documents)} docs → {chroma_collection.count()} chunks")

## 4. Retrieval with Full Provenance

Every retrieved chunk is logged: which query, which paper, what score. Nothing is thrown away.

In [ ]:
FORESIGHT_QUERIES = [
    # Cross-domain disruption
    "What technologies from outside the wireless industry could disrupt spectrum monitoring?",
    "How could advances in biotechnology or materials science affect RF hardware?",
    "What can the wireless industry learn from how other industries were disrupted?",
    # Weak signals
    "What niche or experimental approaches to spectrum sensing exist that nobody scales yet?",
    "What failed or abandoned wireless technologies might become relevant again with new enablers?",
    # Non-obvious second-order effects
    "How will climate change or energy policy affect wireless infrastructure and spectrum use?",
    "What are unintended consequences of satellite mega-constellations for terrestrial spectrum users?",
    "How could open-source hardware movements change the test and measurement industry?",
    # Geopolitical / regulatory wildcards
    "What spectrum policy changes could fundamentally restructure the wireless market?",
    "How do defense and civilian spectrum needs conflict, and what new solutions exist?",
    # Adjacent blind spots
    "How will edge computing and on-device AI reduce the need for traditional network testing?",
    "What role will blockchain or decentralized systems play in spectrum management?",
]

retrieval_log = []
retriever = index.as_retriever(similarity_top_k=RETRIEVAL_TOP_K)

for q in FORESIGHT_QUERIES:
    nodes = retriever.retrieve(q)
    for nws in nodes:
        retrieval_log.append({
            "query": q,
            "node_id": nws.node_id,
            "text": nws.node.text,
            "score": nws.score,
            "paper_title": nws.node.metadata.get("title", ""),
            "paper_id": nws.node.metadata.get("paper_id", ""),
        })

seen_nodes = set()
unique_chunks = []
for rec in retrieval_log:
    if rec["node_id"] not in seen_nodes:
        seen_nodes.add(rec["node_id"])
        unique_chunks.append(rec)

print(f"Retrieved {len(retrieval_log)} total chunks, {len(unique_chunks)} unique")
print(f"From {len(set(r['paper_id'] for r in unique_chunks))} distinct papers")

## 5. Extract Drivers with Source Evidence

Each chunk is tagged with `[CHUNK N]` so the LLM can cite specific passages. Every driver must have at least one verbatim quote.

In [ ]:
from src.models import ObservableDriverExtractionResult

def format_chunk(rec: dict, idx: int) -> str:
    return f'[CHUNK {idx}] (Paper: "{rec["paper_title"]}" | ID: {rec["paper_id"]})\n{rec["text"]}\n[/CHUNK {idx}]'

EXTRACTION_SYSTEM = """You are a technology foresight analyst hunting for BLIND SPOTS in Rohde & Schwarz's strategic planning (spectrum monitoring, T&M, wireless comms, defense electronics).

Your job is NOT to list well-known technology trends. R&S already knows about 5G, MIMO, cognitive radio, Open RAN.

Instead, look for:
- Cross-domain signals: technology from OUTSIDE wireless that could disrupt R&S's world
- Weak signals: early-stage, low-TRL ideas that could become important in 5-10 years
- Non-obvious second-order effects: how a known trend creates unexpected downstream consequences
- Regulatory/geopolitical wildcards: policy shifts that could restructure the market
- Adjacent-field disruptions: what happened in other industries that could repeat here

For EACH driver you MUST:
1. Copy EXACT verbatim text excerpts from the chunks as evidence (do NOT paraphrase)
2. Reference the paper title and ID from the [CHUNK N] headers
3. Explain why R&S might NOT already be tracking this

RULES:
- Every driver MUST have at least one exact quote from the source chunks in chunk_text
- The chunk_text field must be copied verbatim from the source text
- If you cannot find a supporting passage, do NOT extract that driver
- SKIP obvious/mainstream drivers (5G testing, basic MIMO, standard spectrum sensing)
- Prefer surprising, cross-domain, or contrarian findings
- Aim for 5-8 high-quality drivers per batch, not 12 generic ones"""

all_driver_records = []

for batch_start in range(0, len(unique_chunks), EXTRACTION_BATCH_SIZE):
    batch = unique_chunks[batch_start:batch_start + EXTRACTION_BATCH_SIZE]
    context = "\n\n---\n\n".join(format_chunk(rec, i) for i, rec in enumerate(batch))

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM},
            {"role": "user", "content": f"Extract NON-OBVIOUS technology drivers from these research chunks:\n\n{context}"},
        ],
        response_format=ObservableDriverExtractionResult,
        temperature=0.4,
    )

    batch_drivers = response.choices[0].message.parsed.drivers
    for driver in batch_drivers:
        all_driver_records.append({"driver": driver, "batch_chunks": batch})

    print(f"  Batch {batch_start // EXTRACTION_BATCH_SIZE + 1}: {len(batch_drivers)} drivers")

print(f"\nTotal raw drivers: {len(all_driver_records)}")
print(f"Total evidence pieces: {sum(len(r['driver'].evidence) for r in all_driver_records)}")

## 6. Deduplicate Drivers (merge evidence)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

def azure_embed(texts: list[str]) -> np.ndarray:
    all_embeds = []
    for i in range(0, len(texts), 16):
        batch = texts[i:i+16]
        resp = client.embeddings.create(input=batch, model=EMBED_MODEL)
        all_embeds.extend([d.embedding for d in resp.data])
    return np.array(all_embeds)

all_drivers = [r["driver"] for r in all_driver_records]
driver_texts = [f"{d.name}: {d.description}" for d in all_drivers]
driver_embeddings = azure_embed(driver_texts)

clustering = AgglomerativeClustering(n_clusters=None, distance_threshold=0.25, metric="cosine", linkage="average")
labels = clustering.fit_predict(driver_embeddings)

deduped_drivers = []
deduped_embeddings = []
for cluster_id in range(labels.max() + 1):
    members = [i for i, l in enumerate(labels) if l == cluster_id]
    best = max(members, key=lambda i: len(all_drivers[i].evidence))
    representative = all_drivers[best].model_copy()
    seen_evidence = {(e.paper_id, e.chunk_text[:100]) for e in representative.evidence}
    for m in members:
        if m == best:
            continue
        for e in all_drivers[m].evidence:
            key = (e.paper_id, e.chunk_text[:100])
            if key not in seen_evidence:
                seen_evidence.add(key)
                representative.evidence.append(e)
    deduped_drivers.append(representative)
    deduped_embeddings.append(driver_embeddings[best])

deduped_embeddings = np.array(deduped_embeddings)
print(f"Deduplicated: {len(all_drivers)} → {len(deduped_drivers)} unique drivers (threshold=0.25)")
print(f"Total evidence: {sum(len(d.evidence) for d in deduped_drivers)} pieces")

# Show clusters with >1 member
for cluster_id in range(labels.max() + 1):
    members = [i for i, l in enumerate(labels) if l == cluster_id]
    if len(members) > 1:
        print(f"\n  Merged cluster ({len(members)} members):")
        for m in members:
            print(f"    - {all_drivers[m].name}")

## 7. Faithfulness Check (RAGAS-style)

Decompose each driver's description into atomic claims, verify each against source evidence.

In [ ]:
from src.models import FaithfulnessResult

FAITHFULNESS_THRESHOLD = 0.5
faithfulness_results = []

for i, driver in enumerate(deduped_drivers):
    evidence_block = "\n\n".join(
        f'[Source: "{e.paper_title}" ({e.paper_id})]\n{e.chunk_text}' for e in driver.evidence
    )

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a fact-checking assistant. Given a technology driver description and its source evidence, "
                "decompose the description into atomic factual claims and verify each against the evidence.\n\n"
                "For each claim:\n"
                "- If DIRECTLY supported by a source passage, mark supported=true and quote the passage\n"
                "- If NOT supported (extrapolation, opinion, or no matching passage), mark supported=false\n\n"
                "Be strict: only mark a claim as supported if the source text clearly states it."
            )},
            {"role": "user", "content": (
                f"DRIVER: {driver.name}\n"
                f"DESCRIPTION: {driver.description}\n\n"
                f"SOURCE EVIDENCE:\n{evidence_block}"
            )},
        ],
        response_format=FaithfulnessResult,
        temperature=0.1,
    )
    faithfulness_results.append(response.choices[0].message.parsed)
    score = response.choices[0].message.parsed.faithfulness_score
    status = "✓" if score >= FAITHFULNESS_THRESHOLD else "✗ DROPPED"
    print(f"  [{i+1}/{len(deduped_drivers)}] {driver.name}: {score:.0%} {status}")

# Filter out unfaithful drivers
pre_filter = len(deduped_drivers)
faithful_mask = [f.faithfulness_score >= FAITHFULNESS_THRESHOLD for f in faithfulness_results]
deduped_drivers = [d for d, keep in zip(deduped_drivers, faithful_mask) if keep]
deduped_embeddings = deduped_embeddings[[i for i, keep in enumerate(faithful_mask) if keep]]
faithfulness_results = [f for f, keep in zip(faithfulness_results, faithful_mask) if keep]

avg_faith = np.mean([f.faithfulness_score for f in faithfulness_results])
print(f"\nDropped {pre_filter - len(deduped_drivers)} drivers below {FAITHFULNESS_THRESHOLD:.0%} threshold")
print(f"Remaining: {len(deduped_drivers)} drivers, avg faithfulness: {avg_faith:.0%}")

## 8. LLM Surprise Assessment

"Would this surprise an R&S strategist? Why?" — with R&S product/strategy context.

In [ ]:
from src.models import SurpriseAssessment

RS_CONTEXT = """Rohde & Schwarz context:
- Core business: Spectrum monitoring systems, T&M instruments (signal/spectrum analyzers, network analyzers, oscilloscopes), wireless comms testing, defense electronics (EW, SIGINT)
- Known strategic priorities: 5G/6G test solutions, Open RAN testing, satellite comms, AI-assisted signal analysis, cybersecurity
- Customers: Telecom operators, defense/government agencies, aerospace, automotive (radar), semiconductor
- Revenue ~€3B, 14,000 employees, Munich-based, privately held"""

surprise_assessments = []

for i, driver in enumerate(deduped_drivers):
    evidence_summary = "; ".join(f'"{e.paper_title}"' for e in driver.evidence[:5])

    response = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You assess technology drivers for blind spots in R&S strategic planning.\n\n"
                f"{RS_CONTEXT}\n\n"
                "Consider:\n"
                "- Is this already in R&S's known product/technology roadmap?\n"
                "- Does this come from an adjacent field they might not watch?\n"
                "- Could this disrupt their core business model?\n"
                "- Is there a non-obvious second-order effect?"
            )},
            {"role": "user", "content": (
                f"DRIVER: {driver.name}\n"
                f"DESCRIPTION: {driver.description}\n"
                f"EVIDENCE FROM: {evidence_summary}\n"
                f"STEEP: {driver.steep_category} | TRL: {driver.trl_low}-{driver.trl_high}"
            )},
        ],
        response_format=SurpriseAssessment,
        temperature=0.4,
    )
    surprise_assessments.append(response.choices[0].message.parsed)
    sa = response.choices[0].message.parsed
    print(f"  [{i+1}/{len(deduped_drivers)}] {driver.name}: {sa.surprise_rating}/5 {'(blind spot!)' if sa.surprise_rating >= 4 else ''}")

# Also compute geometric surprise for comparison
centroid = deduped_embeddings.mean(axis=0, keepdims=True)
geo_similarities = cosine_similarity(deduped_embeddings, centroid).flatten()
geo_surprise = 1 - geo_similarities

print(f"\nLLM surprise: {sum(1 for s in surprise_assessments if s.surprise_rating >= 4)} drivers rated 4+/5")
print(f"Geometric surprise range: {geo_surprise.min():.3f} — {geo_surprise.max():.3f}")

## 9. Assemble Audit Reports + Export

In [ ]:
from src.models import DriverAuditReport, RetrievalRecord

audit_reports = []

for driver, faith, surprise_llm, surprise_geo in zip(deduped_drivers, faithfulness_results, surprise_assessments, geo_surprise):
    matching_records = []
    for evidence in driver.evidence:
        for rec in retrieval_log:
            if rec["paper_id"] == evidence.paper_id:
                matching_records.append(RetrievalRecord(
                    query=rec["query"],
                    chunk_node_id=rec["node_id"],
                    chunk_text_preview=rec["text"][:200],
                    paper_title=rec["paper_title"],
                    paper_id=rec["paper_id"],
                    similarity_score=rec["score"],
                ))
                break

    audit_reports.append(DriverAuditReport(
        driver=driver,
        retrieval_records=matching_records,
        faithfulness=faith,
        surprise_geometric=float(surprise_geo),
        surprise_llm=surprise_llm,
    ))

output_dir = Path("../data/audit_reports")
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / "poc2c_audit_reports.json", "w") as f:
    json.dump([r.model_dump() for r in audit_reports], f, indent=2, default=str)

print(f"Saved {len(audit_reports)} audit reports to data/audit_reports/poc2c_audit_reports.json")

## 10. Summary Dashboard

In [ ]:
summary_data = []
for report in audit_reports:
    summary_data.append({
        "Driver": report.driver.name,
        "STEEP": report.driver.steep_category,
        "TRL": f"{report.driver.trl_low}-{report.driver.trl_high}",
        "Evidence": len(report.driver.evidence),
        "Faithfulness": f"{report.faithfulness.faithfulness_score:.0%}",
        "Geo Surprise": f"{report.surprise_geometric:.3f}",
        "LLM Surprise": f"{report.surprise_llm.surprise_rating}/5",
        "Known by R&S": report.surprise_llm.known_by_rs,
    })

summary_df = pd.DataFrame(summary_data).sort_values("LLM Surprise", ascending=False)
summary_df

## 11. Geometric vs. LLM Surprise — Where Do They Disagree?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

geo_scores = [r.surprise_geometric for r in audit_reports]
llm_scores = [r.surprise_llm.surprise_rating for r in audit_reports]
names = [r.driver.name for r in audit_reports]
faith_scores = [r.faithfulness.faithfulness_score for r in audit_reports]

scatter = ax.scatter(geo_scores, llm_scores, c=faith_scores, cmap="RdYlGn", s=100, edgecolors="black", linewidths=0.5, vmin=0, vmax=1)
for i, name in enumerate(names):
    ax.annotate(name, (geo_scores[i], llm_scores[i]), fontsize=7, alpha=0.8,
                xytext=(5, 5), textcoords="offset points")

ax.set_xlabel("Geometric Surprise (embedding distance)")
ax.set_ylabel("LLM Surprise Rating (1-5)")
ax.set_title("Geometric vs. LLM Surprise\nColor = Faithfulness (green=grounded, red=ungrounded)")
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1 (expected)", "2", "3 (somewhat)", "4", "5 (blind spot)"])
plt.colorbar(scatter, label="Faithfulness Score")
plt.tight_layout()
plt.show()

## 12. Audit Cards — Full Trail per Driver

In [ ]:
from IPython.display import Markdown, display

sorted_reports = sorted(audit_reports, key=lambda r: r.surprise_llm.surprise_rating, reverse=True)

for report in sorted_reports[:10]:
    d = report.driver
    f = report.faithfulness
    s = report.surprise_llm

    evidence_md = ""
    for i, e in enumerate(d.evidence[:3], 1):
        evidence_md += f'\n{i}. **[{e.paper_title}]** ({e.paper_id})\n   > {e.chunk_text[:300]}{"..." if len(e.chunk_text) > 300 else ""}\n   *Relevance: {e.relevance}*\n'

    unsupported_md = ""
    unsupported_claims = [c for c in f.claims if not c.supported]
    if unsupported_claims:
        unsupported_md = "\n**Unsupported claims:**\n" + "\n".join(f"- ⚠ {c.claim}" for c in unsupported_claims)

    retrieval_md = ""
    for rec in report.retrieval_records[:3]:
        retrieval_md += f"\n- Query: *\"{rec.query}\"* → {rec.paper_title} (score: {rec.similarity_score:.3f})"

    contrarian = f"\n**Contrarian angle:** {s.contrarian_angle}" if s.contrarian_angle else ""

    card = f"""### {d.name}

**STEEP:** {d.steep_category} | **TRL:** {d.trl_low}-{d.trl_high} | **Faithfulness:** {f.faithfulness_score:.0%} | **Surprise:** {s.surprise_rating}/5

{d.description}

**Source Evidence:**
{evidence_md}
**Faithfulness:** {len([c for c in f.claims if c.supported])}/{len(f.claims)} claims supported{unsupported_md}

**Surprise Assessment ({s.surprise_rating}/5):** {s.explanation}

**R&S Relevance:** {s.rs_relevance}

**Known by R&S:** {"Yes" if s.known_by_rs else "No"}{contrarian}

**Retrieval Trail:**{retrieval_md}

---"""

    display(Markdown(card))

## 13. Search Bias Analysis — Which queries produced which drivers?

In [ ]:
query_driver_matrix = np.zeros((len(FORESIGHT_QUERIES), len(audit_reports)))

for j, report in enumerate(audit_reports):
    evidence_paper_ids = {e.paper_id for e in report.driver.evidence}
    for rec in retrieval_log:
        if rec["paper_id"] in evidence_paper_ids:
            q_idx = FORESIGHT_QUERIES.index(rec["query"]) if rec["query"] in FORESIGHT_QUERIES else -1
            if q_idx >= 0:
                query_driver_matrix[q_idx, j] = 1

driver_names = [r.driver.name for r in audit_reports]
fig, ax = plt.subplots(figsize=(max(14, len(driver_names) * 0.6), 8))
sns.heatmap(query_driver_matrix, xticklabels=driver_names, yticklabels=[q[:50] + "..." for q in FORESIGHT_QUERIES],
            cmap="Blues", ax=ax, linewidths=0.5, cbar_kws={"label": "Query contributed to driver"})
ax.set_title("Query → Driver Attribution\nDrivers from a single query = fragility signal")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

single_query_drivers = [driver_names[j] for j in range(len(audit_reports)) if query_driver_matrix[:, j].sum() <= 1]
if single_query_drivers:
    print(f"\n⚠ Drivers sourced from only 1 query (fragile): {single_query_drivers}")

In [ ]:
# Hop attribution: which citation hops contributed to each driver?
paper_hop_map = {p["id"]: p["hop"] for p in papers}

hop_labels = sorted(set(p["hop"] for p in papers))
hop_driver_matrix = np.zeros((len(hop_labels), len(audit_reports)))

for j, report in enumerate(audit_reports):
    for e in report.driver.evidence:
        hop = paper_hop_map.get(e.paper_id, -1)
        if hop in hop_labels:
            hop_driver_matrix[hop_labels.index(hop), j] = 1

fig, ax = plt.subplots(figsize=(max(14, len(driver_names) * 0.6), 4))
sns.heatmap(hop_driver_matrix, xticklabels=driver_names,
            yticklabels=[f"Hop {h} {'(seeds)' if h == 0 else ''}" for h in hop_labels],
            cmap="Oranges", ax=ax, linewidths=0.5, cbar_kws={"label": "Evidence from this hop"})
ax.set_title("Citation Hop → Driver Attribution\nDrivers only from seeds = citation graph didn't help")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.show()

seed_only = [driver_names[j] for j in range(len(audit_reports))
             if hop_driver_matrix[0, j] > 0 and hop_driver_matrix[1:, j].sum() == 0]
deep_only = [driver_names[j] for j in range(len(audit_reports))
             if hop_driver_matrix[0, j] == 0 and hop_driver_matrix[1:, j].sum() > 0]
if seed_only:
    print(f"Drivers from seeds only (no citation expansion): {seed_only}")
if deep_only:
    print(f"Drivers ONLY from citation hops (not in seeds): {deep_only}")